In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURAÇÕES GERAIS
# ============================================================

f0 = 70                 # Frequência original (Hz)
fs = 100                # Frequência de amostragem (Hz)
A = 1.0                 # Amplitude
phi = 0.0               # Fase inicial

T = 1.0                 # Duração da simulação (s)

# Frequência de Nyquist
f_nyquist = fs / 2

# Frequência aparente após o aliasing
f_alias = abs(f0 - fs)


# ============================================================
# 2. GERAÇÃO DO SINAL CONTÍNUO SIMULADO
# ============================================================

# Alta frequência de amostragem para simular o sinal contínuo
fs_continuo = 10000

t_continuo = np.arange(
    0, T, 1/fs_continuo
)

# Senoide original de 70 Hz
x_continuo = A * np.cos(
    2 * np.pi * f0 * t_continuo + phi
)

# Senoide equivalente de 30 Hz
x_alias_continuo = A * np.cos(
    2 * np.pi * f_alias * t_continuo + phi
)


# ============================================================
# 3. REALIZAÇÃO DA AMOSTRAGEM
# ============================================================

N = int(fs * T)

n = np.arange(N)

# Instantes de amostragem
t_amostras = n / fs

# Amostragem da senoide original
x_amostras = A * np.cos(
    2 * np.pi * f0 * t_amostras + phi
)

# Senoide equivalente avaliada nos mesmos instantes
x_alias_amostras = A * np.cos(
    2 * np.pi * f_alias * t_amostras + phi
)


# ============================================================
# 4. REPRESENTAÇÃO DOS SINAIS NO DOMÍNIO DO TEMPO
# ============================================================

plt.figure(figsize=(12, 6))

plt.plot(
    t_continuo,
    x_continuo,
    label="Sinal original (70 Hz)",
    color="blue"
)

plt.plot(
    t_continuo,
    x_alias_continuo,
    "--",
    label="Sinal aparente (30 Hz)",
    color="green"
)

plt.stem(
    t_amostras,
    x_amostras,
    linefmt="red",
    markerfmt="ro",
    basefmt="k-",
    label="Amostras"
)

plt.title("Demonstração do Aliasing no domínio do tempo")
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 5. REPRESENTAÇÃO DA SEQUÊNCIA DISCRETA
# ============================================================

plt.figure(figsize=(12, 5))

plt.stem(
    n,
    x_amostras,
    linefmt="blue",
    markerfmt="bo",
    basefmt="k-"
)

plt.title("Sequência discreta obtida após a amostragem")
plt.xlabel("Índice da amostra (n)")
plt.ylabel("x[n]")

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 6. CÁLCULO DO ESPECTRO
# ============================================================

# Transformada de Fourier
X = np.fft.rfft(x_amostras)

# Eixo de frequências
frequencias = np.fft.rfftfreq(
    N,
    d=1/fs
)

# Magnitude normalizada
magnitude = np.abs(X) / N

# Correção para espectro unilateral
if N % 2 == 0:
    magnitude[1:-1] *= 2
else:
    magnitude[1:] *= 2


# ============================================================
# 7. REPRESENTAÇÃO DO ESPECTRO
# ============================================================

plt.figure(figsize=(12, 5))

plt.stem(
    frequencias,
    magnitude
)

plt.axvline(
    f_alias,
    color="red",
    linestyle="--",
    label=f"Frequência aparente = {f_alias} Hz"
)

plt.axvline(
    f_nyquist,
    color="green",
    linestyle=":",
    label=f"Nyquist = {f_nyquist} Hz"
)

plt.title("Espectro de magnitude do sinal amostrado")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude")

plt.xlim(0, fs/2)

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 8. COMPARAÇÃO DAS FREQUÊNCIAS
# ============================================================

frequencias_comparacao = [f0, f_alias]

plt.figure(figsize=(9, 5))

plt.bar(
    ["Frequência original", "Frequência aparente"],
    frequencias_comparacao
)

plt.axhline(
    f_nyquist,
    color="red",
    linestyle="--",
    label="Frequência de Nyquist"
)

plt.title("Comparação entre frequências")
plt.ylabel("Frequência (Hz)")

plt.legend()
plt.grid(axis="y")
plt.tight_layout()
plt.show()


# ============================================================
# 9. VERIFICAÇÃO MATEMÁTICA DO ALIASING
# ============================================================

# Erro entre as amostras dos dois sinais
erro = np.max(
    np.abs(x_amostras - x_alias_amostras)
)

print("ANÁLISE DO ALIASING")
print("=" * 50)

print(f"Frequência original: {f0} Hz")
print(f"Frequência de amostragem: {fs} Hz")
print(f"Frequência de Nyquist: {f_nyquist} Hz")
print(f"Frequência aparente: {f_alias} Hz")

print("\nVerificação das condições:")

print(
    f"Frequência original > Nyquist: "
    f"{f0 > f_nyquist}"
)

print(
    f"Frequência aparente < Nyquist: "
    f"{f_alias < f_nyquist}"
)

print("\nComparação entre as amostras:")

print(
    f"Erro máximo entre as sequências: "
    f"{erro:.10e}"
)

print(
    f"Sequências equivalentes: "
    f"{np.allclose(x_amostras, x_alias_amostras)}"
)

# Identificação do pico espectral
indice_pico = np.argmax(magnitude)

print("\nAnálise espectral:")

print(
    f"Frequência identificada: "
    f"{frequencias[indice_pico]:.2f} Hz"
)

print(
    f"Magnitude do pico: "
    f"{magnitude[indice_pico]:.4f}"
)

print("\nSimulação concluída!")